# fMRI EDA: overview

Checks and exploration of the preprocessed LEMON resting-state fMRI (159 scans). Each claim is a **check** with an ID (A1, B2, ...) and a PASS / FAIL / INFO status. The explanations are in [FINDINGS.md](FINDINGS.md).
Only group-level files are read here (`results/`, `figures/`).

In [1]:
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())   # works from any folder
import json
from IPython.display import Image, display
RES, FIG = ROOT / "analysis/fmri_eda/results", ROOT / "analysis/fmri_eda/figures"
ck = pd.read_csv(RES / "checks_group.csv")
print(ck.status.value_counts().to_dict())

{'PASS': 19, 'INFO': 19, 'FAIL': 5}


## 1. All checks
FAIL does not mean bad data: A1 is a known header error, and D2/D3, F5 and H4 are explained in FINDINGS.md.

In [2]:
pd.set_option("display.max_colwidth", 120)
ck[["id", "section", "status", "claim", "value"]]

,id,section,status,claim,value
0,A1,Timing,FAIL,Header TR (pixdim[4]) of the preprocessed files,[1.0]
1,A2,Timing,PASS,RepetitionTime in raw-scan bold.json sidecars,{1.4: 227}
2,A3,Timing,PASS,Scanner trigger spacing in physio recording (1 participant),"657 triggers; interval median 1.4000 s, min 1.4000, max 1.4000"
3,A4,Timing,PASS,Time points per scan = acquired volumes - 5 dropped,[652]; 657 - 5 = 652
4,A5,Timing,INFO,Usable scan duration,912.8 s = 15.21 min
5,A6,Timing,INFO,Volumes after resampling to 2 s (as Brain-JEPA training data),456 volumes
6,B1,Geometry,PASS,Header fields identical across scans,"{""shape"": true, ""zooms"": true, ""affine"": true, ""datatype"": true, ""descrip"": true, ""qform_code"": true, ""sform_code"": ..."
7,B2,Geometry,PASS,Grid is the standard FSL MNI152 2 mm grid,"max abs diff 0.0000; shape [91, 109, 91]"
8,B3,Geometry,INFO,No slice-timing info and no scaling in header,"slice_duration 0.0, toffset 0.0, scl_slope None"
9,C1,Values,PASS,No NaN or Inf values,"NaN 0, Inf 0"


## 2. Independent verification
Key numbers recomputed with separate code from the raw files, and one scan processed on two machines.

In [3]:
pd.read_csv(RES / "verification_summary.csv")[["check", "comparisons", "passed", "failed", "max_abs_difference"]]

,check,comparisons,passed,failed,max_abs_difference
0,MeanFD: mean |diff| of columns 10-12 vs QC PDF,10,10,0,6.742836e-04
1,NaN count,10,10,0,0.000000e+00
2,all bold.json TR == 1.4 (count of other values),1,1,0,0.000000e+00
3,brain-like voxel count,10,10,0,0.000000e+00
4,header pixdim[4] (TR field),10,10,0,0.000000e+00
5,header time points,10,10,0,0.000000e+00
6,laptop vs Colab: arrays not bit-identical (count),1,1,0,0.000000e+00
7,laptop vs Colab: gs_lag1_autocorr,1,1,0,1.110223e-16
8,laptop vs Colab: gs_std,1,1,0,0.000000e+00
9,laptop vs Colab: homotopic_corr_mean,1,1,0,0.000000e+00


## 3. Older vs younger (descriptive)

In [4]:
pd.read_csv(RES / "age_group_summary.csv").round(3)

,measure,n_older,n_younger,older_mean,younger_mean,difference,cohens_d,perm_p,n_perm,perm_kind
0,MeanFD (mm),46,113,0.230,0.125,0.105,1.836,0.000,10000,random
1,mean connectivity,46,113,0.167,0.154,0.013,0.206,0.239,10000,random
2,split-half FC reliability,46,113,0.533,0.515,0.019,0.259,0.144,10000,random
3,global signal std,46,113,0.116,0.119,-0.003,-0.119,0.487,10000,random
4,flagged volumes,46,113,5.870,5.496,0.374,0.074,0.675,10000,random


## 4. Figures (linked, not embedded, to keep the notebook small)

**Timing: the true TR is 1.4 s (header says 1.0)**

![fig01_timing_evidence](figures/fig01_timing_evidence.png)

**Signal values are z-scored and near-Gaussian**

![fig02_value_distribution](figures/fig02_value_distribution.png)

**Two kinds of voxels: empty padding and brain**

![fig03_voxel_std_histogram](figures/fig03_voxel_std_histogram.png)

**Where the brain is, and how variable it is**

![fig04_brain_maps](figures/fig04_brain_maps.png)

**Frequency content shaped by the pipeline's band-pass filter**

![fig05_power_spectrum](figures/fig05_power_spectrum.png)

**Correlation falls with distance; left-right mirror coupling**

![fig10_spatial_structure](figures/fig10_spatial_structure.png)

**Connectivity: group matrix, person-specific fingerprint, distance profile**

![fig11_connectivity](figures/fig11_connectivity.png)

**Older vs younger**

![fig12_age_groups](figures/fig12_age_groups.png)